In [1]:
import fastf1
import pandas as pd
import numpy as np

In [4]:
fastf1.Cache.enable_cache("f1_cache")

SEASONS = [2025, 2026]
now = pd.Timestamp.now(tz="UTC")

In [5]:
calendar_records = []

for year in SEASONS:

    schedule = fastf1.get_event_schedule(
        year,
        include_testing=False
    )

    for _, event in schedule.iterrows():

        if event["RoundNumber"] == 0:
            continue

        for i in range(1, 6):

            session_name = event.get(f"Session{i}")
            session_date = event.get(f"Session{i}DateUtc")

            if pd.isna(session_name) or pd.isna(session_date):
                continue

            calendar_records.append({
                "Year": year,
                "RoundNumber": int(event["RoundNumber"]),
                "EventName": event["EventName"],
                "Country": event["Country"],
                "Location": event["Location"],
                "EventDate": event["EventDate"],
                "Session": session_name,
                "SessionDate": event.get(f"Session{i}Date"),
                "SessionDateUTC": session_date
            })

season_calendar_df = pd.DataFrame(calendar_records)

season_calendar_df["SessionDateUTC"] = pd.to_datetime(
    season_calendar_df["SessionDateUTC"],
    utc=True
)

season_calendar_df["SessionStatus"] = season_calendar_df[
    "SessionDateUTC"
].apply(lambda x: "Upcoming" if x > now else "Scheduled/Past")

display(season_calendar_df.head())
print("Calendar shape:", season_calendar_df.shape)

,Year,RoundNumber,EventName,Country,Location,EventDate,Session,SessionDate,SessionDateUTC,SessionStatus
0,2025,1,Australian Grand Prix,Australia,Melbourne,2025-03-16,Practice 1,2025-03-14 12:30:00+11:00,2025-03-14 01:30:00+00:00,Scheduled/Past
1,2025,1,Australian Grand Prix,Australia,Melbourne,2025-03-16,Practice 2,2025-03-14 16:00:00+11:00,2025-03-14 05:00:00+00:00,Scheduled/Past
2,2025,1,Australian Grand Prix,Australia,Melbourne,2025-03-16,Practice 3,2025-03-15 12:30:00+11:00,2025-03-15 01:30:00+00:00,Scheduled/Past
3,2025,1,Australian Grand Prix,Australia,Melbourne,2025-03-16,Qualifying,2025-03-15 16:00:00+11:00,2025-03-15 05:00:00+00:00,Scheduled/Past
4,2025,1,Australian Grand Prix,Australia,Melbourne,2025-03-16,Race,2025-03-16 15:00:00+11:00,2025-03-16 04:00:00+00:00,Scheduled/Past


Calendar shape: (235, 10)


In [12]:
season_calendar_df[(season_calendar_df['Year'] == 2026) & (season_calendar_df['Location'] == 'Las Vegas')]


,Year,RoundNumber,EventName,Country,Location,EventDate,Session,SessionDate,SessionDateUTC,SessionStatus
220,2026,21,Las Vegas Grand Prix,United States,Las Vegas,2026-11-21,Practice 1,2026-11-19 16:30:00-08:00,2026-11-20 00:30:00+00:00,Upcoming
221,2026,21,Las Vegas Grand Prix,United States,Las Vegas,2026-11-21,Practice 2,2026-11-19 20:00:00-08:00,2026-11-20 04:00:00+00:00,Upcoming
222,2026,21,Las Vegas Grand Prix,United States,Las Vegas,2026-11-21,Practice 3,2026-11-20 16:30:00-08:00,2026-11-21 00:30:00+00:00,Upcoming
223,2026,21,Las Vegas Grand Prix,United States,Las Vegas,2026-11-21,Qualifying,2026-11-20 20:00:00-08:00,2026-11-21 04:00:00+00:00,Upcoming
224,2026,21,Las Vegas Grand Prix,United States,Las Vegas,2026-11-21,Race,2026-11-21 20:00:00-08:00,2026-11-22 04:00:00+00:00,Upcoming


In [ ]:
# results_list = []

# for _, event in season_calendar_df.iterrows():

#     if event["SessionDateUTC"] > now:
#         continue

#     try:

#         session = fastf1.get_session(
#             int(event["Year"]),
#             int(event["RoundNumber"]),
#             event["Session"]
#         )

#         session.load(
#             laps=True,
#             telemetry=False,
#             weather=False,
#             messages=False
#         )

#         results = session.results.copy()
#         laps = session.laps.copy()

#         if results.empty:
#             continue

#         # Standardize driver number for merging
#         results["DriverNumber"] = results["DriverNumber"].astype(str)
#         laps["DriverNumber"] = laps["DriverNumber"].astype(str)

#         # Aggregate lap timing by driver
#         lap_summary = laps.groupby("DriverNumber").agg(
#             BestLapTime=("LapTime", "min"),
#             BestSector1=("Sector1Time", "min"),
#             BestSector2=("Sector2Time", "min"),
#             BestSector3=("Sector3Time", "min"),
#             LapsCompleted=("LapNumber", "max")
#         ).reset_index()

#         # Merge driver results with lap timings
#         session_df = results.merge(
#             lap_summary,
#             on="DriverNumber",
#             how="left"
#         )

#         # Add event and session information
#         session_df["Year"] = event["Year"]
#         session_df["RoundNumber"] = event["RoundNumber"]
#         session_df["EventName"] = event["EventName"]
#         session_df["Country"] = event["Country"]
#         session_df["Location"] = event["Location"]
#         session_df["Session"] = event["Session"]

#         results_list.append(session_df)

#         print(
#             f"Collected: {event['Year']} - "
#             f"{event['EventName']} - {event['Session']}"
#         )

#     except Exception as e:

#         print(
#             f"Skipped: {event['Year']} - "
#             f"{event['EventName']} - {event['Session']}: {e}"
#         )


# session_results_df = pd.concat(
#     results_list,
#     ignore_index=True
# ) if results_list else pd.DataFrame()

# print("Results shape:", session_results_df.shape)

# display(session_results_df.head())

In [ ]:
output = Path("season_data")
output.mkdir(exist_ok=True)

season_calendar_df.to_csv(
    output / "season_calendar.csv", index=False
)

session_results_df.to_parquet(
    output / "session_results.parquet", index=False
)

aris_master_df.to_parquet(
    output / "aris_master.parquet", index=False
)

print("All season datasets exported successfully!")

NameError: name 'Path' is not defined

In [ ]:
# print("CALENDAR:", season_calendar_df.shape)
# print("RESULTS:", session_results_df.shape)
# print("MASTER:", aris_master_df.shape)

# print("\nSESSIONS:")
# print(season_calendar_df.groupby(
#     ["Year", "Session"]
# ).size())

# print("\nMASTER COLUMNS:")
# print(aris_master_df.columns.tolist())

CALENDAR: (235, 10)
RESULTS: (0, 0)


NameError: name 'aris_master_df' is not defined

In [16]:
season_calendar_df.to_csv("season_calendar.csv", index=False)